# Step 6D: SCIMAP Phenotyping

This notebook performs cell phenotyping using the **SCIMAP** methodology with a professional, publication-ready structure identical to the Leiden, SOM, and threshold clustering methods. It provides a per-ROI processing workflow with log-transformation, rescaling, and sequential gating strategy.

## Overview

The SCIMAP phenotyping approach:
1. **Data Preparation** - Load batch-corrected marker data  
2. **Per-ROI Processing** - Process each batch/ROI with specific thresholds
3. **Log Transformation** - Apply log1p transformation to marker data
4. **Manual Rescaling** - Scale to [0,1] range using calculated thresholds
5. **Sequential Gating** - Apply phenotype strategy for cell assignment
6. **Quality Control** - Comprehensive validation and outputs

## Input
- Batch-corrected CSV files from Step 5
- ROI-specific threshold files for rescaling
- External phenotype strategy CSV file

## Output
- CSV files with SCIMAP phenotype assignments
- Expression heatmaps for validation
- Quality control visualizations

## Key Features
- **Per-ROI thresholding** - Sample-specific threshold calculation
- **Manual rescaling** - SCIMAP-compatible rescaling methodology
- **Sequential gating** - External strategy file for phenotype assignment
- **Comprehensive outputs** - Compatible with unified phenotyping pipeline

In [ ]:
# Imports & Environment Setup
import os, sys, importlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import time
warnings.filterwarnings('ignore')

# Add src to path for imports
project_root = Path().resolve().parent
src_path = project_root / "src"
sys.path.insert(0, str(src_path))

from spacemap.config import default_config
from spacemap.ph_scimap_utils import manual_rescale_scimap, apply_scimap_phenotypes

try:
    import scimap as sm
    import anndata
    import scanpy as sc
    print("SCIMAP, AnnData, and Scanpy imported successfully")
except ImportError as e:
    print(f"Error: Required libraries not found. Please install scimap, anndata, and scanpy.")
    print(e)

print(f"Project root: {project_root}")
print("Step 6D: SCIMAP Phenotyping Setup Complete")

## Configuration

Configure the SCIMAP phenotyping parameters below. These settings control the per-ROI processing, rescaling, and phenotype assignment strategy.

In [ ]:
# =============================================================================
# CONFIGURATION 
# =============================================================================

# I/O paths

INPUT_DIR = project_root / 'data/'
INPUT_FILE = INPUT_DIR / 'CRC_clusters_neighborhoods_markers.csv'
OUTPUT_DIR = default_config.output_dir / 'phenotyping'

PLOTS_DIR = OUTPUT_DIR / 'scimap/plots'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# External data directories
THRESHOLD_DIR = project_root / "data/thresholds"
PHENOTYPE_STRATEGY_FILE = project_root / "data" / "phenotype" / "Phenotype_Strategy_SCIMAP.csv"

# Column Configuration  
CELL_ID_COLUMN = 'CellID'            # Name of the cell ID column (configurable)
BATCH_COL = 'File Name'              # Column containing ROI/batch identifiers
IMAGE_ID_COL = 'batch_identifier'    # Column name for image/batch identifier in AnnData

# Filtering Configuration
FILTER_CRITERIA = {
    #'tissue': 1,       # Filter to tissue cells (use None to disable)
}

# Threshold Configuration
APPLY_THRESHOLDS = False     # Set to False to skip thresholding and use raw markers
THRESHOLD_STATISTIC = 'median'  # Statistic to use from threshold files ('median', 'min', 'mean')

# SCIMAP parameters (matching reference script)
GATE_VALUE = 0.5  # Threshold for positivity on rescaled data

# Raw markers for phenotyping (matching reference script exactly)
MARKERS_BASE = [
    "CD31", "E-cadherin", "CD20", "CD4", "Pan-Cytokeratin", "CD68", "Podoplanin", "CD11c", "CD8", "HLA-DR",
    "CD3e", "CD45", "CD15", "FoxP3", "PDGFRa", "aSMA", "CD163", "FN1", "S100A9", "CD11b", "CD38",
    "Vimentin"
]

# Processing parameters
GENERATE_PLOTS = True
MAX_PLOT_CELLS = 100000  # Maximum cells for plotting

print("Configuration loaded:")
print(f"  Input file: {INPUT_FILE}")
print(f"  Output dir: {OUTPUT_DIR}")
print(f"  Cell ID column: {CELL_ID_COLUMN}")
print(f"  Batch column: {BATCH_COL}")
print(f"  Image ID column: {IMAGE_ID_COL}")
print(f"  Threshold directory: {THRESHOLD_DIR}")
print(f"  Apply thresholds: {APPLY_THRESHOLDS}")
print(f"  Threshold statistic: {THRESHOLD_STATISTIC}")
print(f"  Phenotype strategy file: {PHENOTYPE_STRATEGY_FILE}")
print(f"  SCIMAP gate value: {GATE_VALUE}")
print(f"  Raw markers: {len(MARKERS_BASE)}")
print(f"  Filter criteria: {FILTER_CRITERIA}")

# Load external phenotype strategy file (required)
if not PHENOTYPE_STRATEGY_FILE.exists():
    raise FileNotFoundError(f"Phenotype strategy file not found: {PHENOTYPE_STRATEGY_FILE}")

print(f"\nLoading phenotype strategy from: {PHENOTYPE_STRATEGY_FILE}")
phenotype_strategy_df = pd.read_csv(PHENOTYPE_STRATEGY_FILE)
print(f"Strategy file loaded with shape: {phenotype_strategy_df.shape}")
print(f"Strategy columns: {list(phenotype_strategy_df.columns)}")

# Display first few rows for debugging
print("\nStrategy file preview:")
print(phenotype_strategy_df.head())

## Step 1: Data Loading and Preparation

Load the batch-corrected data and prepare it for SCIMAP phenotyping analysis.

**Data validation and filtering**:
- Loads unified batch-corrected dataset
- Applies flexible filtering via `FILTER_CRITERIA` dictionary
- Validates cell ID column for proper anchoring
- Verifies availability of threshold files and phenotype strategy

In [ ]:
# =============================================================================
# STEP 1: DATA LOADING AND PREPARATION
# =============================================================================
print("Step 1: Loading data and preparing SCIMAP input...")
start_time = time.time()

# Load unified batch-corrected data
if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Unified input file not found: {INPUT_FILE}")

print(f"Loading unified data from: {INPUT_FILE}")
df = pd.read_csv(INPUT_FILE, low_memory=False)
original_count = len(df)
print(f"Loaded data shape: {df.shape}")

# Validate cell ID column exists for proper anchoring
if CELL_ID_COLUMN not in df.columns:
    raise ValueError(f"Critical: '{CELL_ID_COLUMN}' column missing from input data")

print(f"Cell ID validation: {df[CELL_ID_COLUMN].nunique():,} unique cell_ids out of {len(df):,} rows")

# Apply flexible filtering criteria
for column, value in FILTER_CRITERIA.items():
    if value is not None:  # Only apply filter if value is not None
        if column in df.columns:
            before = len(df)
            mask = df[column] == value
            df = df[mask].copy()
            filtered_count = len(df)
            print(f" {column} filter applied ({column}=={value}): {filtered_count:,}/{before:,} rows retained")
        else:
            print(f"Warning: '{column}' column not found - skipping filter")

# Keep original dataframe intact to preserve spatial coordinates
original_df = df.copy()

# Check available batches
batch_ids = df[BATCH_COL].unique().tolist()
print(f"\nFound {len(batch_ids)} batches for processing:")
print(f"Batch IDs: {batch_ids[:5]}..." if len(batch_ids) > 5 else f"Batch IDs: {batch_ids}")

# Verify threshold directory exists
if not THRESHOLD_DIR.exists():
    raise FileNotFoundError(f"Threshold directory not found: {THRESHOLD_DIR}")

# Verify markers are available
available_markers = [m for m in MARKERS_BASE if m in df.columns]
if not available_markers:
    raise ValueError('No markers found in data')

total_time = time.time() - start_time
print(f"\nStep 1 Summary:")
print(f" Original rows: {original_count:,}")
print(f" After filtering: {len(df):,}")
print(f" Batches to process: {len(batch_ids)}")
print(f" Available markers: {len(available_markers)}")
print(f" Cell ID column: {CELL_ID_COLUMN} with {df[CELL_ID_COLUMN].nunique():,} unique values")
print(f" Processing time: {total_time:.1f}s")
print(f" Ready for per-ROI SCIMAP processing")

## Step 2: Per-ROI SCIMAP Phenotyping

Process each ROI with the full SCIMAP workflow: threshold calculation, log transformation, rescaling, and phenotype assignment.

**Per-ROI Processing Workflow**:
- **Threshold Loading**: Load ROI-specific polygon thresholds from CSV files
- **Median Calculation**: Calculate median thresholds across polygon gates
- **AnnData Creation**: Convert data to SCIMAP-compatible format
- **Log Transformation**: Apply log1p transformation to marker data
- **Manual Rescaling**: Scale to [0,1] range using calculated thresholds
- **Sequential Gating**: Apply phenotype strategy for cell assignment

In [ ]:
# =============================================================================
# STEP 2: PER-ROI SCIMAP PHENOTYPING
# =============================================================================
print("Step 2: Per-ROI SCIMAP phenotyping...")
start_time = time.time()

phenotyped_dfs = []
all_batch_ids = df[BATCH_COL].unique()

print(f"Processing {len(all_batch_ids)} batches for SCIMAP phenotyping...")

for i, batch_id in enumerate(all_batch_ids):
    print(f"\n--- Processing batch {i+1}/{len(all_batch_ids)}: {batch_id} ---")
    
    # Get batch data
    batch_df = df[df[BATCH_COL] == batch_id].copy()
    
    # Find corresponding threshold file (matching threshold notebook logic)
    batch_prefix = batch_id.split('.ome')[0] if '.ome' in batch_id else batch_id
    threshold_filename = f"{batch_prefix}.ome.tiff_-_Merged_Image.csv"
    threshold_path = THRESHOLD_DIR / threshold_filename
    
    # Handle threshold processing based on configuration
    if APPLY_THRESHOLDS:
        if not threshold_path.exists():
            print(f"  ERROR: Threshold file not found: {threshold_filename}")
            # Set default phenotype for this batch and continue
            batch_df['phenotype_scimap'] = 'unclassified'
            phenotyped_dfs.append(batch_df)
            continue
        
        # Load and process threshold file
        try:
            threshold_df = pd.read_csv(threshold_path)
            print(f"  Successfully loaded threshold file: {threshold_filename}")
            
            # Verify Marker column exists
            if 'Marker' not in threshold_df.columns:
                if len(threshold_df.columns) > 0:
                    marker_col_name = threshold_df.columns[0]
                    threshold_df.rename(columns={marker_col_name: 'Marker'}, inplace=True)
                    print(f"  Renamed '{marker_col_name}' column to 'Marker'")
                else:
                    raise ValueError("No columns found in threshold file")
            
            # Find polygon columns and calculate median thresholds
            polygon_cols = [col for col in threshold_df.columns if col.startswith('Polygon_')]
            if not polygon_cols:
                raise ValueError("No 'Polygon_*' columns found")
            
            # Filter to known markers and calculate thresholds
            threshold_df = threshold_df[threshold_df['Marker'].isin(MARKERS_BASE)].copy()
            if threshold_df.empty:
                raise ValueError("No matching markers found in threshold file")
            
            # Calculate median threshold for each marker
            threshold_results = {}
            for _, row in threshold_df.iterrows():
                marker = row['Marker']
                polygon_values = row[polygon_cols].values
                polygon_values = polygon_values[~pd.isna(polygon_values)]
                
                if len(polygon_values) > 0:
                    if THRESHOLD_STATISTIC == 'median':
                        median_threshold = np.median(polygon_values)
                    elif THRESHOLD_STATISTIC == 'mean':
                        median_threshold = np.mean(polygon_values)
                    elif THRESHOLD_STATISTIC == 'min':
                        median_threshold = np.min(polygon_values)
                    else:
                        median_threshold = np.median(polygon_values)
                    threshold_results[marker] = median_threshold
            
            if not threshold_results:
                raise ValueError("No valid thresholds calculated")
            
            print(f"  Calculated thresholds for {len(threshold_results)} markers using {THRESHOLD_STATISTIC} statistic")
            
            # Create log-transformed thresholds for SCIMAP
            log_thresholds = {marker: np.log1p(threshold) for marker, threshold in threshold_results.items()}
            final_gates = pd.Series(log_thresholds)
            
        except Exception as e:
            print(f"  ERROR: Processing failed: {e}")
            batch_df['phenotype_scimap'] = 'unclassified'
            phenotyped_dfs.append(batch_df)
            continue
    else:
        print(f"  Skipping threshold processing - using raw markers directly")
        final_gates = None  # Will be ignored in rescaling
    
    # Find common markers between data and thresholds (or use all if no thresholds)
    markers_in_data = [m for m in MARKERS_BASE if m in batch_df.columns]
    
    if APPLY_THRESHOLDS and final_gates is not None:
        markers_with_thresholds = list(final_gates.index)
        final_markers = sorted(list(set(markers_in_data) & set(markers_with_thresholds)))
        
        if not final_markers:
            raise ValueError("No common markers found between data and thresholds")
        
        final_gates = final_gates[final_markers]
        print(f"  Processing {len(final_markers)} common markers with thresholds")
    else:
        final_markers = markers_in_data
        print(f"  Processing {len(final_markers)} markers without thresholds")
    
    try:
        # Create AnnData object for SCIMAP
        adata = anndata.AnnData(batch_df[final_markers].astype(np.float32))
        adata.obs_names = batch_df.index.astype(str)
        
        # Add spatial coordinates if available
        if 'X_centroid' in batch_df.columns and 'Y_centroid' in batch_df.columns:
            adata.obsm['spatial'] = batch_df[['X_centroid', 'Y_centroid']].values
        
        # Add observation data (excluding marker columns)
        obs_cols = [col for col in batch_df.columns if col not in final_markers]
        adata.obs = batch_df[obs_cols].copy()
        adata.obs[IMAGE_ID_COL] = batch_id
        
        # SCIMAP workflow: log transform, rescale, phenotype
        sc.pp.log1p(adata)
        adata_scaled = adata.copy()
        
        if APPLY_THRESHOLDS:
            # Use threshold-based scaling
            adata_scaled.X = manual_rescale_scimap(adata, final_gates, apply_thresholds=True)
        else:
            # Use direct 0.5-1.0 scaling without thresholds
            print(f"  Skipping threshold files - using direct scaling to 0.5-1.0 range")
            adata_scaled.X = manual_rescale_scimap(adata, apply_thresholds=False)
        
        # Apply phenotyping using strategy
        sm.tl.phenotype_cells(adata_scaled, phenotype=phenotype_strategy_df, gate=GATE_VALUE, label="phenotype_scimap")
        
        # Get phenotype results and standardize unknown labels
        phenotype_results = adata_scaled.obs['phenotype_scimap'].values
        phenotype_results = np.where(phenotype_results == 'Unknown', 'unclassified', phenotype_results)
        
        batch_df['phenotype_scimap'] = phenotype_results
        phenotyped_dfs.append(batch_df)
        print(f"  -> Done ({len(batch_df)} cells)")
        
    except Exception as e:
        print(f"  ERROR: Processing failed: {e}")
        batch_df['phenotype_scimap'] = 'unclassified'
        phenotyped_dfs.append(batch_df)

# Combine results from all batches
if phenotyped_dfs:
    df_scimap = pd.concat(phenotyped_dfs, ignore_index=True)
    
    # Standardize any remaining unknown labels
    if 'phenotype_scimap' in df_scimap.columns:
        unknown_count = (df_scimap['phenotype_scimap'] == 'Unknown').sum()
        if unknown_count > 0:
            df_scimap['phenotype_scimap'] = df_scimap['phenotype_scimap'].replace('Unknown', 'unclassified')
            print(f"Standardized {unknown_count:,} 'Unknown' labels to 'unclassified'")
    
    processing_time = time.time() - start_time
    print(f"\nStep 2 Summary:")
    print(f" Successfully processed {len(all_batch_ids)} batches")
    print(f" Total cells processed: {len(df_scimap):,}")
    print(f" Processing time: {processing_time:.1f}s")
    print(f" Cell ID column preserved: {df_scimap[CELL_ID_COLUMN].nunique():,} unique values")
    
    # Display phenotype distribution
    phenotype_counts = df_scimap['phenotype_scimap'].value_counts()
    print(f"\nPhenotype distribution:")
    for phenotype, count in phenotype_counts.head(10).items():
        percentage = (count / len(df_scimap)) * 100
        print(f"  {phenotype}: {count:,} ({percentage:.1f}%)")
    if len(phenotype_counts) > 10:
        print(f"  ... and {len(phenotype_counts) - 10} more phenotypes")
        
else:
    print("\nNo batches were successfully processed")
    df_scimap = original_df.copy()
    df_scimap['phenotype_scimap'] = 'unclassified'

## Step 3: Expression Heatmap Generation

Generate heatmap visualization of mean marker expression per phenotype for quality control.

In [ ]:
# =============================================================================
# STEP 3: EXPRESSION HEATMAP GENERATION
# =============================================================================
print("Step 3: Generating expression heatmap...")

if GENERATE_PLOTS and 'df_scimap' in globals() and len(df_scimap) > 0:
    # Filter to only cells with valid phenotypes for heatmap
    plot_data = df_scimap[df_scimap['phenotype_scimap'] != 'unclassified'].copy()
    
    if len(plot_data) == 0:
        print("  No cells with valid phenotypes found for heatmap")
    else:
        print(f"  Creating heatmap with {len(plot_data):,} cells and {len(plot_data['phenotype_scimap'].unique())} phenotypes")
        
        # Use available markers for heatmap
        markers_for_plot = [m for m in MARKERS_BASE if m in plot_data.columns]
        
        if len(markers_for_plot) == 0:
            print("  No marker columns found for heatmap")
        else:
            try:
                # Create professional heatmap with Z-scores
                plt.figure(figsize=(14, max(6, 0.4 * len(plot_data['phenotype_scimap'].unique()))))
                
                # Calculate mean expression per phenotype
                phenotype_means = plot_data.groupby('phenotype_scimap')[markers_for_plot].mean()
                
                # Z-score normalize across phenotypes for each marker
                heatmap_data = phenotype_means.apply(lambda x: (x - x.mean()) / x.std() if x.std() > 0 else 0, axis=0)
                
                # Create heatmap with BWR colormap (matching other methods)
                sns.heatmap(
                    heatmap_data,
                    cmap='bwr',  # Blue-White-Red colormap
                    center=0,
                    annot=False,
                    cbar_kws={'label': 'Z-score Expression', 'shrink': 0.8},
                    xticklabels=True,
                    yticklabels=True,
                    linewidths=0.1,
                    linecolor='white'
                )
                
                plt.title('SCIMAP Phenotyping: Marker Expression Profile', fontsize=14, pad=20)
                plt.xlabel('Markers', fontsize=12)
                plt.ylabel('Phenotypes', fontsize=12)
                plt.xticks(rotation=45, ha='right')
                plt.yticks(rotation=0)
                plt.tight_layout()
                
                # Save heatmap
                heatmap_path = PLOTS_DIR / 'phenotype_marker_heatmap_scimap.png'
                plt.savefig(heatmap_path, dpi=300, bbox_inches='tight')
                plt.show()
                print(f"  Heatmap saved to: {heatmap_path}")
                
            except Exception as e:
                print(f"  Could not generate heatmap: {e}")
else:
    print("Plot generation disabled or no cells were phenotyped")

## Step 4: Apply Phenotypes and Save Results

Apply the phenotype assignments to the dataset and save final results with proper data preservation.

In [ ]:
# =============================================================================
# STEP 4: APPLY PHENOTYPES AND SAVE RESULTS
# =============================================================================
print("Step 4: Applying SCIMAP phenotypes and saving results...")

# FIXED: Use proper apply function instead of manual pandas merge
# This ensures compatibility with other phenotyping methods and proper cell_id anchoring
if 'df_scimap' in globals() and len(df_scimap) > 0:
    # Filter out any normalized columns that might have slipped in
    clean_data = df_scimap[[c for c in df_scimap.columns if not c.endswith('_norm')]].copy()
    
    # Extract phenotype assignments
    phenotype_assignments = clean_data['phenotype_scimap'].copy()
    
    # Apply SCIMAP phenotypes using standardized function
    result = apply_scimap_phenotypes(
        data=clean_data,
        phenotype_assignments=phenotype_assignments,
        output_base_dir=OUTPUT_DIR,
        method_prefix='scimap',
        cell_id_col=CELL_ID_COLUMN
    )

    print(f"\nStep 4 Results:")
    print(f" Unified file: {result['unified_file']}")
    print(f" Summary file: {result['summary_file']}")
    print(f" Mapping file: {result['mapping_file']}")

    print(f"\nPhenotype Summary:")
    summary_display = result['summary_df'].head(10)
    for _, row in summary_display.iterrows():
        phenotype = row.name
        count = int(row['cell_count'])
        pct = row['percentage']
        print(f" {phenotype}: {count:,} cells ({pct:.1f}%)")
    
    if len(result['summary_df']) > 10:
        print(f" ... and {len(result['summary_df']) - 10} more phenotypes")

    # Validation: Check unified file integrity
    unified_df = result['data']
    print(f"\nData Integrity Check:")
    print(f" Original cells: {len(original_df):,}")
    print(f" Unified output cells: {len(unified_df):,}")
    print(f" Cell ID preservation: {unified_df[CELL_ID_COLUMN].nunique():,} unique IDs")
    
    if 'phenotype_scimap' in unified_df.columns:
        print(f" SCIMAP phenotype column added successfully")
        unassigned = unified_df['phenotype_scimap'].isnull().sum()
        if unassigned > 0:
            print(f" Warning: {unassigned} cells have no phenotype assignment")
        else:
            print(f" All cells successfully assigned phenotypes")
    else:
        print(" Warning: phenotype_scimap column not found in output")
        
    print(f"\nSCIMAP phenotyping complete. Results saved to: {result['unified_file']}")
    
else:
    print("Error: No SCIMAP results available for integration")
    print("Please ensure Step 2 completed successfully")

## Summary

This notebook performs SCIMAP-based phenotyping with a professional, publication-ready structure identical to the Leiden, SOM, and threshold clustering methods.

### Workflow Steps
1. **Data Preparation** - Load batch-corrected data with flexible filtering
2. **Per-ROI Processing** - Process each ROI with SCIMAP workflow (thresholds, log-transform, rescale, phenotype)
3. **Expression Heatmap** - Visualization of phenotype marker profiles
4. **Apply Phenotypes** - Generate unified output with preserved data integrity
5. **Batch Export** - Optional individual batch file generation  
6. **Cache Cleanup** - Memory and temporary file cleanup

### Key Outputs
**Main Results** (stored in `phenotyping/`)
- `all_samples_combined_corrected_phenotyped.csv` - Unified dataset with SCIMAP phenotypes
- `scimap_phenotype_summary.csv` - Phenotype statistics and cell counts
- `scimap_phenotype_mapping.csv` - Phenotype to cell count mapping

**Visualizations** (stored in `phenotyping/scimap/plots/`)
- Phenotype expression heatmaps with BWR colormap matching other methods

### Key Features
- **Per-ROI thresholding** - Sample-specific threshold calculation from polygon files
- **SCIMAP compatibility** - Uses proven SCIMAP log-transform and rescaling methodology
- **External strategy** - Configurable phenotype strategy file for flexible gating
- **Sequential gating** - SCIMAP phenotype_cells with gate threshold control
- **Data integrity** - Preserves spatial coordinates and cell ID anchoring
- **Professional workflow** - Matches Leiden/SOM structure and output format

### Method Highlights
- Log1p transformation of marker expression data
- Manual rescaling to [0,1] range using calculated median thresholds
- External phenotype strategy CSV for reproducible gating definitions
- Per-ROI processing maintains spatial heterogeneity
- Compatible output format for integration with other phenotyping methods
- BWR colormap visualization matching other notebooks

### Technical Notes
- Requires external `Phenotype_Strategy_SCIMAP.csv` file for gating strategy
- Processes data per ROI/batch to account for technical variation
- Uses SCIMAP's proven manual_rescale methodology
- Standardizes 'Unknown' labels to 'unclassified' for consistency
- Cell_id anchoring ensures proper data alignment across methods

### Next Steps
The unified output file is ready for downstream spatial analysis and can be combined with results from other phenotyping methods (Leiden, SOM, threshold-based) for comprehensive validation and analysis.

In [ ]:
# =============================================================================
# STEP 5: OPTIONAL BATCH EXPORT
# =============================================================================
import re

# Export configuration
EXPORT_ALL_BATCHES = False       # Set True to export every batch
EXPORT_PREFIXES = ['P11_roi_1']         # Prefixes to match when EXPORT_ALL_BATCHES is False

# Setup batch export directory
BATCH_EXPORT_DIR = OUTPUT_DIR / 'scimap' / 'batch'
BATCH_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Step 5: Optional batch export...")

# Load unified file if not already in memory
if 'result' in globals() and isinstance(result, dict) and 'unified_file' in result:
    unified_path = result['unified_file']
    unified_df = result['data']
else:
    unified_path = OUTPUT_DIR / 'all_samples_combined_corrected_phenotyped.csv'
    if not unified_path.exists():
        print(f"Unified file not found: {unified_path}")
        print("Run Step 4 first to generate unified output file.")
        unified_df = None

if unified_df is not None and BATCH_COL in unified_df.columns:
    # Determine target batches
    batches = unified_df[BATCH_COL].dropna().astype(str).unique().tolist()
    
    if EXPORT_ALL_BATCHES:
        target_batches = batches
        print(f"Exporting ALL {len(target_batches)} batches...")
    else:
        target_batches = []
        for prefix in EXPORT_PREFIXES:
            matched = [b for b in batches if b.startswith(prefix)]
            target_batches.extend(matched)
        
        # Remove duplicates while preserving order
        seen = set()
        target_batches = [b for b in target_batches if not (b in seen or seen.add(b))]
        print(f"Exporting {len(target_batches)} batches matching prefixes {EXPORT_PREFIXES}")

    if not target_batches:
        print('No batches match the export criteria.')
    else:
        # Export each batch
        for batch_id in target_batches:
            batch_df = unified_df[unified_df[BATCH_COL].astype(str) == batch_id].copy()
            
            # Create safe filename
            safe_name = re.sub(r'[\\/*?:"<>|\s]+', '_', batch_id)
            output_file = BATCH_EXPORT_DIR / f'{safe_name}_phenotyped_scimap.csv'
            
            batch_df.to_csv(output_file, index=False)
            print(f" Exported {batch_id}: {len(batch_df):,} cells → {output_file.name}")
        
        print(f"\nBatch export complete. Files saved to: {BATCH_EXPORT_DIR}")
else:
    print("Batch export skipped - no unified file or batch column available")

print("Step 5 complete.")

## Step 6: Cleanup Cache

Clean up any temporary or cache files to ensure fresh computation on next run. This prevents cache inconsistencies and ensures reproducibility.

In [ ]:
# =============================================================================
# STEP 6: CLEANUP CACHE
# =============================================================================
print("Step 6: Cleaning up cache...")

# No specific cache files to clean for SCIMAP (it doesn't use normalization cache)
# But we can clean up any temporary AnnData objects or other intermediate files

# Clean up intermediate variables from memory
cleanup_vars = ['df_scimap', 'phenotyped_dfs', 'all_batch_ids']
cleaned_count = 0

for var_name in cleanup_vars:
    if var_name in globals():
        del globals()[var_name]
        cleaned_count += 1

print(f"Cleaned up {cleaned_count} intermediate variables from memory")

# Optionally clean up any temporary files in plots directory (keeping actual plots)
temp_files = []
if PLOTS_DIR.exists():
    temp_files = [f for f in PLOTS_DIR.iterdir() if f.suffix in ['.tmp', '.temp']]
    for temp_file in temp_files:
        temp_file.unlink()

if temp_files:
    print(f"Deleted {len(temp_files)} temporary files from plots directory")
else:
    print("No temporary files found to clean up")

print("Cache cleanup complete.")